# A3.1 · Default-deny on the tool call

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A2.8 · An audit trail the workload cannot forge](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**.

| | |
|---|---|
| Tools used | OPA / Rego, SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Evaluate the same call under allow-by-default and deny-by-default policy and compare what gets through.

**Why a security engineer needs it.** Allow-by-default authorization is defeated by any argument the model can be persuaded to produce. The control it builds is: policy evaluated per call on (identity, tool, arguments, resource), denying unless a rule permits.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Identity has already failed. Something untrusted is in the context and the agent has decided to call a tool. The tool call is the last place a decision can still be made on facts rather than intent — this SPIFFE ID, this tool, this resource, this verb — and a policy written one notch vaguer than that cannot express the distinction the attack turns on.

> **At CyberTravels.** The last place a decision about that refund rests on facts rather than on intent. Identity has already failed, an injected instruction is in the context, and the tool call is where CyberTravels can still say no. R1, R3.

## 2 · The framework

```
   untrusted text in context ---> agent decides to call a tool
                                             |
                                    +--------v---------+
                                    |  policy decision |
                                    |  DEFAULT: DENY   |
                                    +--------+---------+
                                             |
                        allow only on facts: identity, scope,
                        resource, provenance of the motivating span

   the last point where a decision rests on facts rather than on intent
```

**Mitigates: T2 Tool Misuse · T3 Privilege Compromise · T6 Intent Breaking.**

The tool call is the moment text becomes consequence. It is also the last point
where a decision can be made on **facts** — this identity, this tool, these
arguments, this resource — rather than on intent, which nobody can read.

**Start from the identity, and be very specific about it.** Not "the Workflow
Agent"; the SPIFFE ID A2.3 issued —
`spiffe://cybertravels.com/ns/prod/sa/workflow-agent` — and against it, an
entitlement written at full resolution: which tools, on which resources, with
which verbs. Everything else in this lesson is a consequence of writing the
entitlement down at that resolution. A policy phrased one notch vaguer cannot
express the distinction the attack turns on, and the vagueness is invisible
until it is exploited.

Default-deny then means the absence of a rule is a refusal. That sounds like a
detail and it is the entire control, because it changes what a mistake costs.
Under allow-by-default, a permission somebody forgot to restrict is available to
an attacker. Under deny-by-default, a permission somebody forgot to grant is a
broken feature — which someone reports on Monday morning, loudly, and which
harms nobody.

The policy takes four inputs and all four matter:

- **identity** — the attested workload identity, from A2.3
- **tool** — which capability
- **arguments** — the actual values, not the schema
- **resource** — which specific thing

Dropping the fourth is the most common weakening. `run_query` permitted for the
Workflow Agent is not the same as `run_query` permitted *on the bookings table*,
and A1.5 was the difference between those two sentences. The verb is the second
most common: `charge_card` entitled for payments is not `refund` entitled for
payments, and R1 is the entire distance between them.

This does not stop the agent being persuaded. It stops persuasion mattering,
which is a better place to stand.

> **What this control closes.**
>
> Stands on the edge every topology shares: `agent_runtime -> tools`. Persuasion still happens; it just stops reaching anything.

## 3 · Proving the baseline is default-deny, as a skill

Writing the entitlement is one job; showing that CyberTravels' running role *is* the entitlement is another, and it is the one an auditor asks for. The procedure reads every inline and attached policy for the baseline, then measures granted-but-unused permissions against what the audit trail observed — which only means anything if the trail is intact, so incomplete coverage is a finding rather than a clean pass. This is the file in this repository:

### The skill — [`skills/attestation/iam-least-privilege-verifier/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/iam-least-privilege-verifier/SKILL.md)

```yaml
name: iam-least-privilege-verifier
description: >-
  Prove a deployment's role baseline is default-deny and quantify granted-
  but-unused permissions against observed usage. Use to evidence least
  privilege, to find excess permissions on an agent role, or when asked
  whether a deployment's IAM posture supports a default-deny claim.
allowed-tools: Bash, Read
```

# Iam Least Privilege Verifier

**Controls:** Control 1 — default-deny and least privilege

## Confidence: HIGH

This is one of the controls that is genuinely provable at runtime. Policy
documents are readable, and usage data turns "least privilege" from an
assertion into a measured delta.

## When to use this
Whenever a default-deny or least-privilege claim is about to be made, and again
whenever the role changes. It needs observed usage as well as the policy, so it
runs against a deployment that has been live long enough to have a usage
window — a fresh deployment shows every permission as unused and the result
means nothing.

## Procedure

1. **Establish the default-deny baseline.** Read every inline and attached
   policy. The baseline fails if any of these are present:
   - `Action: "*"` or `Resource: "*"` in an Allow statement
   - broad managed policies such as administrator or power-user equivalents
   - a wildcard principal on a trust policy

2. **Measure excess.** Compare granted permissions against observed usage from
   the access-analysis and last-accessed data. Count actions, roles, keys and
   passwords idle for at least the tracking period (configurable 1–180 days;
   default 90).

3. **Generate the least-privilege diff.** Policy generation derived from actual
   activity produces a candidate policy; the diff against what is granted is
   the excess-permission finding, expressed concretely rather than as a score.

4. **Check external access.** External-access findings must be zero, or each
   one must map to an approved exception.

## Example

**Input** — the fixture committed at the top of [`scripts/iam_least_privilege_verifier.py`](scripts/iam_least_privilege_verifier.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
   charge_card payments:booking          ['CHARGE']
   run_query   table:bookings            ['SELECT', 'UPDATE']
   send_email  domain:cybertravels.com   ['*']
allow-by-default:
   run_query   table:bookings                SELECT ALLOW run_query on table:bookings permits SELECT
   run_query   table:customer_pii            SELECT ALLOW allowed by default
   charge_card payments:booking              REFUND deny  REFUND not permitted on payments:booking (only ['CHARGE'])
   send_email  domain:archive.evil.example   *      ALLOW allowed by default
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "deployment_id": "str",
  "role_arn": "str",
  "default_deny_verified": true,
  "wildcard_findings": [{"policy": "str", "statement": "str"}],
  "excess_permission_count": 0,
  "unused": [{"type": "action|role|key|password", "name": "str", "idle_days": 0}],
  "generated_policy_diff": "str",
  "external_access_findings": 0,
  "tracking_period_days": 90,
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Incomplete audit-trail coverage.** If the trail is not intact, "unused" is
  unreliable and the verdict must be `PARTIAL`. Policy generation can miss
  legitimately-used-but-rare actions — an annual disaster-recovery permission
  looks identical to dead permission over a 90-day window.
- **Counting managed-policy names instead of effective actions.** Two policies
  can grant the same action; the union is what matters.
- **Treating a low excess count as a pass** while a wildcard is present. The
  baseline check is a gate, not a contributor to a score.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/attestation/iam-least-privilege-verifier/scripts/iam_least_privilege_verifier.py
SCRIPT = "skills/attestation/iam-least-privilege-verifier/scripts/iam_least_privilege_verifier.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The skill loads and reports its shape. Two of its failure modes are the ones this lesson is about: counting managed-policy *names* instead of effective actions, and reading a low excess count as a pass while a wildcard sits in the policy — a wildcard is not a large number of permissions, it is an unbounded one.

## Your turn

Take one tool policy you have and check whether it names the resource *and* the verb. If it grants `run_query` rather than `SELECT on these tables`, it cannot express the difference that A1.5 and R1 both turn on.

---

**Next → [A3.2 · Sandboxed execution](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*